# Predicting Annual Wages of Full-Time Workers in Texas
### Machine Learning and Data Analytics (6COSC017C) — Coursework 1
**Student ID:** 00017665

**Data:** American Community Survey (ACS) 2024 1-Year Public Use Microdata Sample (PUMS), Texas person file — U.S. Census Bureau (public domain).
**Task:** supervised **regression** — predict a full-time employee's annual wage (`WAGP`) from education, occupation, industry, job and demographic characteristics.

**Contents**
1. Data loading and instance selection
2. Exploratory Data Analysis (EDA)
3. Data preparation
4. Model training
5. Model evaluation

## 0. Setup
All libraries are pinned in `pyproject.toml` / `uv.lock`, so the notebook can be reproduced with `uv sync`.
A fixed random seed is used everywhere randomness appears.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Project paths (the notebook lives in notebooks/, data in data/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_FILE = ROOT / "data" / "raw" / "psam_p48.csv"
PROCESSED_FILE = ROOT / "data" / "processed" / "acs_tx_2024_fulltime.csv"

# Helper module with the decoding tables for the PUMS numeric codes
sys.path.append(str(ROOT / "src"))
from mlda_cw1 import codes

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)   # 2 decimals, thousands separator

# Chart style: seaborn's clean theme + two colours used throughout the notebook
sns.set_theme(style="whitegrid")
BLUE, ORANGE = "#2a78d6", "#eb6834"

## 1. Data loading and instance selection

### 1.1 Source
The ACS is the largest household survey in the United States (~1% of the population each year).
The PUMS file contains **one row per person** with anonymised answers to the questionnaire.
We use the 2024 Texas person-level file, downloaded from
<https://www2.census.gov/programs-surveys/acs/data/pums/2024/1-Year/> (`csv_ptx.zip` → `psam_p48.csv`).
Texas was chosen because it is the second-largest state with a diverse economy (energy, technology, healthcare, agriculture), giving a large and varied sample.

### 1.2 Which columns are loaded and why
The raw file has 286 columns; most are irrelevant (housing allocation flags, 80 replicate weights, etc.).
We load only the variables that plausibly explain wages **and are known before the wage is observed**:

| Group | Variables | Meaning |
|---|---|---|
| Identifiers | `SERIALNO`, `SPORDER` | household id + person number (unique person key) |
| Target | `WAGP` | wages or salary income in the past 12 months (USD) |
| Adjustment | `ADJINC` | factor that converts income to 2024 dollars |
| Weight | `PWGTP` | person weight (how many people each row represents) |
| Human capital | `AGEP`, `SCHL`, `FOD1P` | age, educational attainment, field of bachelor's degree |
| Job | `OCCP`, `INDP`, `COW`, `WKHP`, `WKWN` | occupation, industry, class of worker, usual hours/week, weeks worked |
| Commute | `JWMNP`, `JWTRNS` | travel time to work (min), means of transport |
| Demographics | `SEX`, `MAR`, `RAC1P`, `HISP`, `NATIVITY`, `ENG`, `DIS`, `MIL` | used for EDA and fairness checks |
| Geography | `PUMA` | Public Use Microdata Area (~217 regions in Texas) |
| Filter only | `ESR` | employment status |

**Deliberately excluded — target leakage:** `PERNP` (total earnings) and `PINCP` (total income) *contain* `WAGP`, so using them would let the model "see the answer".

In [ ]:
LOAD_COLS = [
    "SERIALNO", "SPORDER", "PUMA", "ADJINC", "PWGTP",
    "AGEP", "SEX", "SCHL", "FOD1P", "MAR", "RAC1P", "HISP", "NATIVITY", "ENG", "DIS", "MIL",
    "COW", "ESR", "OCCP", "INDP", "WKHP", "WKWN", "JWMNP", "JWTRNS",
    "WAGP",
]

if RAW_FILE.exists():
    raw_columns = pd.read_csv(RAW_FILE, nrows=0).columns        # header only -> total column count
    raw = pd.read_csv(RAW_FILE, usecols=LOAD_COLS, low_memory=False)
    print(f"Raw file: {len(raw):,} rows x {len(raw_columns)} columns "
          f"-> loaded {raw.shape[1]} selected columns")
else:
    raw = None
    print("Raw file not found - the processed dataset will be loaded in step 1.4.")

### 1.3 Instance selection (filtering the population)
The research question is about **wages of full-time employees**, so rows outside that population are removed.
Each condition has a clear reason:

| Condition | Reason |
|---|---|
| `AGEP` 18–64 | working-age adults; excludes children and most retirees |
| `ESR` ∈ {1, 2} | civilian employed (at work / with a job but not at work) |
| `COW` ∈ {1…5} | employees of private firms or government; the self-employed earn *business income*, not wages |
| `WKHP` ≥ 35 | full-time (U.S. Bureau of Labor Statistics definition) — removes part-time effects |
| `WAGP` > 0 | a wage must exist to be predicted |

The table below is a **filtering funnel**: it shows how many people remain after each step.
(The last condition removes no extra rows, because every full-time employee in the file reported a wage, but it is kept as a safeguard.)

In [ ]:
if raw is not None:
    steps = [
        ("All persons in file",             pd.Series(True, index=raw.index)),
        ("Age 18-64",                       raw["AGEP"].between(18, 64)),
        ("Civilian employed (ESR 1-2)",     raw["ESR"].isin([1, 2])),
        ("Employee, not self-employed",     raw["COW"].between(1, 5)),
        ("Full-time (35+ hours/week)",      raw["WKHP"] >= 35),
        ("Positive wage income",            raw["WAGP"] > 0),
    ]
    mask = pd.Series(True, index=raw.index)
    funnel = []
    for name, condition in steps:
        mask &= condition                                     # conditions are applied cumulatively
        funnel.append((name, int(mask.sum())))
    funnel = pd.DataFrame(funnel, columns=["Step", "Rows remaining"])
    funnel["% of file"] = funnel["Rows remaining"] / len(raw) * 100
    display(funnel)

    df = raw.loc[mask].drop(columns=["ESR"]).reset_index(drop=True)   # ESR is constant after filtering

### 1.4 Saving the working dataset
The filtered file (~92 thousand rows, 24 columns) is small enough for GitHub, the Streamlit app and the submission zip,
while the 206 MB raw file stays local (it is listed in `.gitignore`).
If the raw file is absent, the notebook simply loads this saved dataset, so every later step is reproducible.

In [ ]:
if raw is not None:
    PROCESSED_FILE.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(PROCESSED_FILE, index=False)
    print(f"Saved {PROCESSED_FILE.name}: {df.shape[0]:,} rows x {df.shape[1]} columns, "
          f"{PROCESSED_FILE.stat().st_size / 1e6:.1f} MB")
    del raw                                                   # free memory
else:
    df = pd.read_csv(PROCESSED_FILE)
    print(f"Loaded {PROCESSED_FILE.name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

df.head()

### 1.5 Data type conversion
Code columns that contain blanks are read by pandas as decimals (`float`, e.g. occupation `4640.0`), because the classic integer type cannot store a missing value.
They are converted to pandas' **nullable integer** type `Int64`, which keeps whole numbers and still allows missing values (`<NA>`).

In [ ]:
INT_COLS = ["COW", "ENG", "JWMNP", "JWTRNS", "MAR", "MIL", "SCHL", "WKHP", "WKWN",
            "FOD1P", "INDP", "OCCP"]
df[INT_COLS] = df[INT_COLS].astype("Int64")
df["WAGP"] = df["WAGP"].astype(float)          # target stays a continuous number
df.head()

## 2. Exploratory Data Analysis
EDA answers four questions before any modelling: *what does the data look like, how clean is it,
how is the target distributed, and which variables are related to it?* Each finding below feeds a concrete decision in Section 3.

### 2.1 Structure and variable types

In [ ]:
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
print(df.dtypes.value_counts().to_string())

All columns except the id `SERIALNO` are stored as **numbers**, but most of them are actually **categories encoded as numbers**
(e.g. `SEX` 1 = male, 2 = female; `OCCP` 2205 = postsecondary teachers). Treating these codes as quantities would be meaningless
(occupation 2205 is not "more" than 1005), so each variable is assigned its true statistical type:

In [ ]:
var_types = pd.DataFrame({
    "variable": ["WAGP", "AGEP", "WKHP", "WKWN", "JWMNP",
                 "SCHL", "ENG",
                 "SEX", "NATIVITY", "DIS",
                 "OCCP", "INDP", "COW", "MAR", "RAC1P", "HISP", "MIL", "JWTRNS", "FOD1P", "PUMA",
                 "SERIALNO", "SPORDER", "PWGTP", "ADJINC"],
    "type": ["numerical (continuous) - TARGET", "numerical (discrete)", "numerical (discrete)",
             "numerical (discrete)", "numerical (discrete)",
             "categorical (ordinal)", "categorical (ordinal)",
             "categorical (binary)", "categorical (binary)", "categorical (binary)"]
            + ["categorical (nominal)"] * 10
            + ["identifier", "identifier", "survey weight", "constant adjustment factor"],
})
var_types.groupby("type")["variable"].apply(", ".join).to_frame()

To make the analysis readable, the codes are **decoded** into labels using the official PUMS data dictionary
(see `src/mlda_cw1/codes.py`). Detailed occupation (~520 codes) and industry (~250 codes) are grouped into
the Census major groups (25 and 17). These label columns are used for EDA only; the formal encoding is done in Section 3.

In [ ]:
eda = df.copy()
eda["log_wage"]   = np.log(eda["WAGP"])                  # log of the target (justified in 2.5)
eda["sex"]        = eda["SEX"].map(codes.SEX)
eda["education"]  = codes.edu_level(eda["SCHL"])
eda["occupation"] = codes.occupation_group(eda["OCCP"])
eda["industry"]   = codes.industry_sector(eda["INDP"])
eda["employer"]   = eda["COW"].map(codes.COW)
eda["marital"]    = eda["MAR"].map(codes.MAR)
eda["nativity"]   = eda["NATIVITY"].map(codes.NATIVITY)
eda["race_eth"]   = codes.race_ethnicity(eda["RAC1P"], eda["HISP"])
eda["disability"] = eda["DIS"].map(codes.DIS)
eda["military"]   = eda["MIL"].map(codes.MIL)
eda["transport"]  = eda["JWTRNS"].map(codes.JWTRNS)
eda[["WAGP", "sex", "education", "occupation", "industry", "employer"]].head()

### 2.2 Missing values and duplicates
Missing data must be understood *before* it is treated: values can be missing at random, or missing **for a structural reason**
that itself carries information.

In [ ]:
missing = pd.DataFrame({"missing": df.isna().sum()})
missing["pct"] = missing["missing"] / len(df) * 100
missing = missing[missing["missing"] > 0].sort_values("pct", ascending=False)
display(missing)

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(missing.index[::-1], missing["pct"][::-1], color=BLUE, height=0.6)
for y, v in enumerate(missing["pct"][::-1]):
    ax.text(v + 1, y, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, 100); ax.set_xlabel("% of rows missing"); ax.grid(axis="y", visible=False)
ax.set_title("Missing values by column")
plt.tight_layout(); plt.show()

In [ ]:
# Why are values missing? Cross-check each column against the variable that explains it.
print("FOD1P missing vs. education (field of degree exists only for bachelor's or higher):")
display(pd.crosstab(eda["education"], eda["FOD1P"].isna(), normalize="index").round(3)
          .rename(columns={False: "has FOD1P", True: "FOD1P missing"}))

print("\nJWMNP (commute time) missing vs. means of transport:")
transport = eda["transport"].fillna("Not at work last week")
display(pd.crosstab(transport, eda["JWMNP"].isna())
          .rename(columns={False: "has JWMNP", True: "JWMNP missing"})
          .sort_values("has JWMNP", ascending=False).head(6))

print(f"\nENG missing: {df['ENG'].isna().mean():.1%} -> in PUMS 'not applicable' means the person speaks only English at home")

**Finding:** none of the missing values are random errors — all are **structurally missing**:

* `ENG` (66%) is blank for people who speak **only English** → it will become a category "Only English", not be imputed.
* `FOD1P` (57%) is blank for everyone **without a bachelor's degree** → new category "No degree".
* `JWMNP` (14%) is blank for people who **work from home** or were **not at work** that week → commute time 0 / separate flag.
* `JWTRNS` (1%) is blank for those **not at work last week** → category "Not at work".

Dropping these rows would delete two-thirds of the data and bias the sample, so in Section 3 they are filled with meaningful values.

In [ ]:
dup_people = df.duplicated(subset=["SERIALNO", "SPORDER"]).sum()
dup_rows   = df.drop(columns=["SERIALNO", "SPORDER"]).duplicated().sum()
print(f"Duplicate person keys (SERIALNO + SPORDER): {dup_people}")
print(f"Rows identical in every other column:       {dup_rows}")

There are no duplicated persons. The few rows that coincide on every attribute belong to **different people**
(different household/person ids) who happen to share the same coded answers — they are genuine observations and are kept.

### 2.3 Summary statistics — numerical variables
Measures of **central tendency** (mean, median) and **dispersion** (variance, standard deviation, IQR) describe each variable;
**skewness** and **kurtosis** describe the shape. Comparing mean and median is a quick test for skew: if mean ≫ median, the
distribution has a long right tail.

In [ ]:
num_cols = ["WAGP", "AGEP", "WKHP", "WKWN", "JWMNP"]
summary = df[num_cols].describe().T
summary["variance"] = df[num_cols].var()
summary["IQR"]      = summary["75%"] - summary["25%"]
summary["skewness"] = df[num_cols].skew()
summary["kurtosis"] = df[num_cols].kurt()
summary = summary[["count", "mean", "50%", "std", "variance", "min", "25%", "75%", "max",
                   "IQR", "skewness", "kurtosis"]].rename(columns={"50%": "median"})
summary

In [ ]:
# The survey weight PWGTP says how many Texans each respondent represents.
# Comparing the weighted and unweighted mean checks whether the sample is representative.
weighted_mean = np.average(df["WAGP"], weights=df["PWGTP"])
print(f"Unweighted mean wage: ${df['WAGP'].mean():,.0f}")
print(f"Weighted mean wage:   ${weighted_mean:,.0f}")
print(f"The sample represents {df['PWGTP'].sum():,} full-time employees in Texas")

**Findings:**
* **Wages** — mean ≈ \$79k but median \$60k; skewness ≈ 3.9 and very high kurtosis → strongly **right-skewed** with a heavy tail of high earners.
* **Hours** — median 40 (the standard full-time week); values up to 99 are rare but possible (e.g. medical residents).
* **Weeks worked** — over 90% worked all 52 weeks; part-year workers will earn less for the same "rate", so `WKWN` is an important predictor.
* The **weighted** mean (using `PWGTP`) is about 6% lower than the unweighted one: the survey slightly over-samples
  higher earners. This matters for estimating *population* averages, but the ML task is to predict an individual's wage *from their characteristics*,
  so the models are trained on unweighted rows (standard ML practice). This is noted as a limitation in the report.

### 2.4 Summary statistics — categorical variables
For categorical variables the relevant statistics are **frequencies** (mode and class balance) and the **median wage per category**,
which gives a first idea of which categories pay more.

In [ ]:
def category_profile(col):
    # Count, share and median wage for each category of a column
    g = eda.groupby(col, observed=True)["WAGP"]
    out = pd.DataFrame({"count": g.size(), "share %": g.size() / len(eda) * 100, "median wage": g.median()})
    return out.sort_values("median wage", ascending=False)

for col in ["education", "sex", "employer", "race_eth"]:
    print(f"\n=== {col} ===")
    display(category_profile(col))

In [ ]:
occ = category_profile("occupation").sort_values("count")
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(occ.index, occ["count"], color=BLUE, height=0.65)
ax.set_xlabel("Number of workers in sample"); ax.grid(axis="y", visible=False)
ax.set_title("Workers by occupation group")
plt.tight_layout(); plt.show()

**Findings:** the most common education level is a bachelor's degree, and the sample is 54% male / 46% female.
Occupation groups are **imbalanced** (over 10,000 management and office workers vs. about 250 in extraction or farming) —
rare groups will be handled carefully when encoding, and a stratified split by wage level will keep the test set representative.

### 2.5 Distribution of the target variable
The shape of the target decides the modelling approach. A strongly skewed target makes squared-error models such as Linear Regression
dominated by a few very high earners, so a **log transformation** is tested.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(df["WAGP"] / 1000, bins=80, color=BLUE, edgecolor="white", linewidth=0.3)
axes[0].axvline(df["WAGP"].mean() / 1000, color=ORANGE, lw=2, label=f"mean ${df['WAGP'].mean()/1000:,.0f}k")
axes[0].axvline(df["WAGP"].median() / 1000, lw=2, ls="--", label=f"median ${df['WAGP'].median()/1000:,.0f}k")
axes[0].set_title(f"Annual wage (skewness {df['WAGP'].skew():.2f})")
axes[0].set_xlabel("Wage, thousand USD"); axes[0].set_ylabel("Number of workers"); axes[0].legend()

axes[1].hist(eda["log_wage"], bins=80, color=BLUE, edgecolor="white", linewidth=0.3)
axes[1].set_title(f"log(annual wage) (skewness {eda['log_wage'].skew():.2f})")
axes[1].set_xlabel("log(wage)"); axes[1].set_ylabel("Number of workers")
plt.tight_layout(); plt.show()

In [ ]:
# Q-Q plot: if the log-wage were normal, the points would lie on the straight line
fig, ax = plt.subplots(figsize=(5, 4))
stats.probplot(eda["log_wage"], dist="norm", plot=ax)
ax.set_title("Q-Q plot of log(wage)")
plt.tight_layout(); plt.show()

**Finding:** raw wages are heavily right-skewed (skewness ≈ 3.9). After the log transform the bulk of the distribution is close to normal
(skewness ≈ −0.55) — the Q-Q plot follows the line in the centre. The remaining deviation is a **left tail of implausibly low wages**
and a **flat cap at the top**, examined next. **Decision:** the models will predict `log(WAGP)`; predictions are converted back
to dollars with `exp()` for interpretation.

### 2.6 Outliers and erroneous values
Outliers are checked with the **IQR rule** (outside Q1 − 1.5·IQR … Q3 + 1.5·IQR, applied to log-wage because the raw scale is skewed)
and with **domain logic**. Following the lectures, each outlier is classified as an *error* (to correct/remove) or a *real rare value* (to keep).

In [ ]:
q1, q3 = eda["log_wage"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"IQR bounds on log-wage: [{low:.2f}, {high:.2f}]  ->  "
      f"[${np.exp(low):,.0f}, ${np.exp(high):,.0f}]")
print(f"Below lower bound: {(eda['log_wage'] < low).sum():,}   above upper bound: {(eda['log_wage'] > high).sum():,}")

# Domain check 1: implied hourly wage = annual wage / (hours per week x weeks worked)
eda["hourly_wage"] = eda["WAGP"] / (eda["WKHP"] * eda["WKWN"])
print(f"\nImplied hourly wage < $7.25 (federal minimum wage): {(eda['hourly_wage'] < 7.25).sum():,} rows")
print(f"Implied hourly wage < $2.00 (almost certainly an error): {(eda['hourly_wage'] < 2).sum():,} rows")

# Domain check 2: top-coding. The Census replaces the highest wages with one capped value for anonymity.
top = df["WAGP"].max()
print(f"\nTop-coded value: ${top:,.0f}  ({(df['WAGP'] == top).sum():,} rows share exactly this value)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
sns.boxplot(x=eda["log_wage"], ax=axes[0], color=BLUE, fliersize=1.5)
axes[0].axvline(low, color=ORANGE, ls="--", lw=1.5); axes[0].axvline(high, color=ORANGE, ls="--", lw=1.5)
axes[0].set_title("Box plot of log(wage) with IQR fences"); axes[0].set_xlabel("log(wage)")

axes[1].hist(np.log10(eda["hourly_wage"]), bins=80, color=BLUE, edgecolor="white", linewidth=0.3)
axes[1].axvline(np.log10(7.25), color=ORANGE, lw=2, label="$7.25 federal minimum")
axes[1].set_xticks(np.log10([1, 7.25, 25, 100, 1000]), ["$1", "$7.25", "$25", "$100", "$1,000"])
axes[1].set_title("Implied hourly wage (log scale)"); axes[1].set_xlabel("USD per hour"); axes[1].legend()
plt.tight_layout(); plt.show()

**Findings and decisions for Section 3:**
* **Low tail = data errors.** Some full-time, full-year workers report a few hundred dollars per year, i.e. an implied hourly rate of cents.
  This is impossible for an employee in Texas (minimum wage \$7.25), so it reflects reporting/recording errors.
  These rows will be removed using a documented threshold on the implied hourly wage.
* **High tail = real values, but censored.** About 1.4% of rows share the identical value \$601,000 — the Census **top-code** (the true
  wage is *at least* this). They are real high earners and will be kept; the cap is noted as a limitation.
* Applying the IQR rule blindly would also delete legitimate high earners (doctors, executives), which is why domain logic is used instead.

### 2.7 Correlation analysis
Different variable types need different correlation measures (Week 2):

* **Pearson** — linear relation between numerical variables;
* **Spearman** — monotonic relation, valid for **ordinal** variables such as education (`SCHL`);
* **Cramér's V** — association between two **nominal** variables (0 = none, 1 = perfect);
* **Point-biserial** — relation between a **binary** variable and a **numerical** target (e.g. sex and wage).

Correlations serve two purposes: finding the strongest predictors of wage, and detecting **multicollinearity** (predictors that duplicate each other).

In [ ]:
corr_cols = ["log_wage", "SCHL", "AGEP", "WKHP", "WKWN", "JWMNP"]
pearson  = eda[corr_cols].corr(method="pearson")
spearman = eda[corr_cols].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, mat, name in zip(axes, [pearson, spearman], ["Pearson", "Spearman"]):
    sns.heatmap(mat, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, ax=ax)
    ax.set_title(f"{name} correlation matrix")
plt.tight_layout(); plt.show()

**Findings:**
* By **Spearman**, education (`SCHL`, ρ ≈ 0.47) is clearly the strongest numerical predictor of wage, followed by hours (0.28), age (0.24) and weeks worked (0.22).
* By **Pearson**, weeks worked looks strongest (r ≈ 0.38). This is driven by the low-wage tail: the few part-year workers with very small annual wages
  pull the linear correlation up. Spearman, which uses ranks, is robust to this — a good illustration of why both measures are reported.
* Education's Spearman (0.47) is much higher than its Pearson (0.34): the relation is **monotonic but not linear** (each extra level adds more money at the top).
* Predictors are almost uncorrelated with each other (all |r| ≤ 0.11), so there is **no multicollinearity** among numerical features.
* Commute time is only weakly related to wage.

In [ ]:
def cramers_v(x, y):
    # Cramer's V from the chi-square statistic of the contingency table
    table = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    r, k = table.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

# Point-biserial correlation for the binary variables (1 = second category, 0 = first)
for col, value in [("sex", "Female"), ("nativity", "Foreign-born"), ("disability", "With disability")]:
    binary = (eda[col] == value).astype(int)
    r_pb, p_value = stats.pointbiserialr(binary, eda["log_wage"])
    print(f"{col:<11} ({value} = 1): r_pb = {r_pb:+.3f}")

cat_cols = ["occupation", "industry", "education", "employer", "sex", "marital",
            "race_eth", "nativity", "disability", "military", "transport"]

In [ ]:
cv = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
for a in cat_cols:
    for b in cat_cols:
        cv.loc[a, b] = 1.0 if a == b else cramers_v(eda[a].astype(str), eda[b].astype(str))

fig, ax = plt.subplots(figsize=(8.5, 7))
mask = np.triu(np.ones_like(cv, dtype=bool))      # show each pair once, hide the trivial diagonal
sns.heatmap(cv, mask=mask, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, square=True, ax=ax)
ax.set_title("Cramér's V between categorical variables")
plt.tight_layout(); plt.show()

**Findings:**
* **Point-biserial:** being female is associated with a lower log-wage (r_pb ≈ −0.16) — a weak but clear raw gap. Being foreign-born (−0.03)
  or having a disability (−0.06) shows almost no linear link on its own.
* For the multi-category variables, the **median wage tables (2.4)** and **box plots (2.8)** compare the groups directly: occupation and education
  separate wages the most, which matches labour-economics theory.
* The strongest associations between predictors are **race/ethnicity ↔ nativity** (V ≈ 0.53), **occupation ↔ sex** (0.47), **occupation ↔ industry** (0.46)
  and **employer type ↔ industry** (0.46). They are related but far from duplicates (V < 0.6), so all are kept. Tree-based models handle such overlap well,
  and the overlap is moderate enough not to distort the linear model.

### 2.8 Wages across key groups
Visual comparisons show *how* the strongest predictors relate to wages — including non-linear shapes that a single correlation number hides.

In [ ]:
edu = eda.groupby("education", observed=True)["WAGP"].median() / 1000
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(edu.index.astype(str), edu.values, color=BLUE, width=0.6)
for x, v in enumerate(edu.values):
    ax.text(x, v + 2, f"${v:,.0f}k", ha="center", fontsize=9)
ax.set_ylabel("Median wage, thousand USD"); ax.grid(axis="x", visible=False)
ax.set_title("Median annual wage by education level")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
order = eda.groupby("occupation")["log_wage"].median().sort_values().index
fig, ax = plt.subplots(figsize=(8, 8))
sns.boxplot(data=eda, y="occupation", x="WAGP", order=order, color=BLUE, fliersize=0,
            ax=ax, log_scale=True, width=0.6)
ax.set_xlim(5_000, 700_000)
ax.set_xticks([10_000, 30_000, 100_000, 300_000], ["$10k", "$30k", "$100k", "$300k"])
ax.set_xlabel("Annual wage (log scale)"); ax.set_ylabel(""); ax.grid(axis="y", visible=False)
ax.set_title("Wage distribution by occupation group")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Age: scatter of a random 5,000-person sample + the median wage at each age
sample = eda.sample(5000, random_state=RANDOM_STATE)
axes[0].scatter(sample["AGEP"], sample["log_wage"], s=5, alpha=0.3, color=BLUE)
age_med = eda.groupby("AGEP")["log_wage"].median()
axes[0].plot(age_med.index, age_med.values, color=ORANGE, lw=2.5, label="median")
axes[0].set_ylim(8, 13.5); axes[0].set_xlabel("Age"); axes[0].set_ylabel("log(wage)")
axes[0].set_title("Wage rises with age, then flattens"); axes[0].legend(loc="lower right")

# Hours: median wage for each band of usual weekly hours
bands = pd.cut(eda["WKHP"], [34, 39, 40, 44, 49, 54, 59, 99],
               labels=["35-39", "40", "41-44", "45-49", "50-54", "55-59", "60+"])
hrs = eda.groupby(bands, observed=True)["WAGP"].median() / 1000
axes[1].bar(hrs.index.astype(str), hrs.values, color=BLUE, width=0.6)
axes[1].set_xlabel("Usual hours worked per week"); axes[1].set_ylabel("Median wage, thousand USD")
axes[1].set_title("Longer hours, higher pay"); axes[1].grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

In [ ]:
gap = (eda.groupby(["education", "sex"], observed=True)["WAGP"].median().unstack() / 1000)
x = np.arange(len(gap))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.2, gap["Male"], width=0.38, color=BLUE, label="Male")
ax.bar(x + 0.2, gap["Female"], width=0.38, color=ORANGE, label="Female")
ax.set_xticks(x, gap.index.astype(str), rotation=20, ha="right")
ax.set_ylabel("Median wage, thousand USD"); ax.grid(axis="x", visible=False); ax.legend()
ax.set_title("Median wage by education and sex")
plt.tight_layout(); plt.show()

gap["female/male ratio"] = gap["Female"] / gap["Male"]
gap.round(2)

**Findings:**
* The median wage rises with every education level up to professional degrees (\$135k); a bachelor's degree roughly **doubles** pay compared with no high-school diploma
  (\$75k vs \$35k). Doctorates (\$105k) earn less than professional degrees (doctors, lawyers), so education is ordinal but **not linear** in its effect.
* Occupation separates wages strongly (management, computing, engineering and healthcare practitioners at the top; food service and personal care at the bottom).
* The **age–wage profile is concave** (fast growth from 18 to about 35, then a plateau) — a linear model needs help to capture it
  (e.g. an age² feature), while tree models learn it automatically.
* Women earn less than men **at every education level** in this raw comparison (female/male median ratio 0.62–0.80). Part of the gap is explained by occupation and hours,
  which the models control for; this is discussed in the ethics section.

### 2.9 EDA summary → decisions for data preparation

| Finding | Decision in Section 3 |
|---|---|
| Categorical variables stored as numeric codes | decode and encode properly (ordinal for education, one-hot for nominal groups) |
| `ENG`, `JWMNP`, `JWTRNS` missing for structural reasons | fill with meaningful values, no row deletion |
| `FOD1P` missing for everyone without a degree, ~170 codes | drop — its information largely overlaps with education and occupation |
| Hundreds of detailed occupation/industry codes | group into major groups (high-cardinality reduction) |
| Target strongly right-skewed | model `log(WAGP)` |
| Implausibly low wages (implied hourly rate far below minimum wage) | remove as data errors with a documented threshold |
| Top-coded wages at \$601k | keep, note as a limitation |
| `ADJINC` constant factor | convert wages to 2024 dollars |
| Concave age profile | add `age²` feature for the linear model |
| Numerical features on different scales (age vs. hours vs. minutes) | standardise for Linear Regression and KNN |
| Weak correlations between predictors | no feature needs to be dropped for multicollinearity |

## 3. Data preparation
This section turns the EDA findings into a clean, model-ready dataset. Every step is justified by a finding from Section 2.
The order follows the lectures (Week 3): **fix errors → handle missing values → engineer features → split → encode and scale**.
Encoding and scaling are learned **only on the training set** (inside a `Pipeline`) to avoid data leakage.

In [ ]:
data = df.copy()
print(f"Starting point: {len(data):,} rows")

### 3.1 Correcting errors and standardising units
1. **Inflation adjustment.** ACS respondents are interviewed across the year and report income for the *previous* 12 months.
   The Census provides `ADJINC` to put all incomes into **2024 dollars**: `wage = WAGP × ADJINC / 1,000,000`.
2. **Impossible wages.** The implied hourly wage is `wage / (hours per week × weeks worked)`. The U.S. federal minimum wage is **\$7.25/hour**,
   so a full-time employee earning less than that is not legally possible — these values are reporting errors and are removed.

In [ ]:
data["wage"] = data["WAGP"] * data["ADJINC"] / 1_000_000
print(f"ADJINC factor: {data['ADJINC'].iloc[0] / 1_000_000}")

data["hourly_wage"] = data["wage"] / (data["WKHP"] * data["WKWN"])
errors = data["hourly_wage"] < 7.25
print(f"Rows with implied hourly wage below $7.25: {errors.sum():,} ({errors.mean():.1%})")

data = data[~errors]
print(f"Rows after removing errors: {len(data):,}")

**Target variable.** As shown in 2.5, wages are strongly right-skewed, so the models predict `log_wage = log(wage)`.

In [ ]:
data["log_wage"] = np.log(data["wage"])
print(f"Skewness of wage:     {data['wage'].skew():.2f}")
print(f"Skewness of log_wage: {data['log_wage'].skew():.2f}")

After removing the errors, the log-wage is almost perfectly symmetric (skewness close to 0).

### 3.2 Handling missing values
All missing values are **structural** (Section 2.2), so they are filled with the value that the blank actually means — no imputation by mean or median is needed:

| Column | Blank means | Filled with |
|---|---|---|
| `ENG` (English ability, 1 = very well … 4 = not at all) | speaks only English | **0** — keeps the scale ordinal: 0 is the best English |
| `JWMNP` (commute minutes) | works from home or was not at work | **0** minutes |
| `JWTRNS` (means of transport) | not at work last week | used only to create the `work_from_home` flag below |

In [ ]:
data["ENG"] = data["ENG"].fillna(0)
data["JWMNP"] = data["JWMNP"].fillna(0)

### 3.3 Feature engineering
New features are created from existing columns to give the models information in a more useful form:

| New feature | How | Why |
|---|---|---|
| `occupation` | ~520 `OCCP` codes → 25 major groups | too many rare codes; groups are meaningful and stable |
| `industry` | ~250 `INDP` codes → 17 sectors | same reason |
| `employer` | decode `COW` (private, non-profit, government) | readable categories |
| `marital` | decode `MAR` | readable categories |
| `age_sq` | `AGEP²` | lets the linear model capture the concave age profile (2.8) |
| `work_from_home` | 1 if `JWTRNS` = 11, else 0 | remote workers earn more on average (2.7) |

`hourly_wage` is **not** used as a feature: it is computed *from* the wage, so it would leak the answer.

In [ ]:
data["occupation"] = codes.occupation_group(data["OCCP"])
data["industry"]   = codes.industry_sector(data["INDP"])
data["employer"]   = data["COW"].map(codes.COW)
data["marital"]    = data["MAR"].map(codes.MAR)
data["age_sq"]     = data["AGEP"] ** 2
data["work_from_home"] = (data["JWTRNS"].fillna(0) == 11).astype(int)   # blank = not at work, not WFH

print("Missing values in new columns:")
print(data[["occupation", "industry", "employer", "marital"]].isna().sum().to_string())

### 3.4 Feature selection
The final feature set contains only variables that describe a person's **skills and job**. Protected characteristics
(sex, race/ethnicity, nativity, disability) are **deliberately excluded** from the model: a wage model used for salary benchmarking should not
set pay based on who someone is. These columns are kept aside to check the model's **fairness** in Section 5.

Other dropped columns: identifiers (`SERIALNO`, `SPORDER`), the weight `PWGTP`, the constant `ADJINC`, the raw codes replaced by groups
(`OCCP`, `INDP`, `COW`, `MAR`, `JWTRNS`), `FOD1P` (see 2.9), `MIL` (very weak link with wage, 2.7) and `PUMA` (217 small regions;
left out to keep the model simple — noted as a limitation).

In [ ]:
NUM_FEATURES = ["AGEP", "age_sq", "SCHL", "WKHP", "WKWN", "JWMNP", "ENG", "work_from_home"]
CAT_FEATURES = ["occupation", "industry", "employer", "marital"]
TARGET = "log_wage"
FAIRNESS_COLS = ["SEX", "RAC1P", "HISP"]      # not used for training, only for the fairness check

model_data = data[NUM_FEATURES + CAT_FEATURES + [TARGET, "wage"] + FAIRNESS_COLS].reset_index(drop=True)
model_data[NUM_FEATURES] = model_data[NUM_FEATURES].astype(float)   # scikit-learn works with plain float columns
print(f"Model dataset: {model_data.shape[0]:,} rows, {len(NUM_FEATURES) + len(CAT_FEATURES)} features")
print(f"Missing values left: {model_data.isna().sum().sum()}")
model_data.head()

### 3.5 Train/test split
The data is split **before** any encoding or scaling is learned, so the test set stays completely unseen (no data leakage).

* **80% train / 20% test.** The test set (~18k people) is large enough for reliable evaluation.
* **No separate validation set:** hyperparameters will be tuned with **5-fold cross-validation on the training set** (Section 4),
  which uses the data more efficiently.
* **Stratified by wage level.** Wages have extreme values, so the target is cut into 10 equal-sized bins (`pd.qcut`) and the split keeps
  the same share of each bin in train and test (Week 5: stratified split for regression with extreme target values).

In [ ]:
from sklearn.model_selection import train_test_split

wage_bins = pd.qcut(model_data[TARGET], q=10, labels=False)
train_df, test_df = train_test_split(model_data, test_size=0.2, random_state=RANDOM_STATE,
                                     stratify=wage_bins)

X_train, y_train = train_df[NUM_FEATURES + CAT_FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[NUM_FEATURES + CAT_FEATURES],  test_df[TARGET]

print(f"Train: {len(X_train):,} rows   Test: {len(X_test):,} rows")
print(f"Median wage - train: ${train_df['wage'].median():,.0f}   test: ${test_df['wage'].median():,.0f}")

The median wage is almost identical in both sets, confirming that the split is representative.

### 3.6 Encoding and scaling (preprocessing pipeline)
* **Categorical features → One-hot encoding.** Occupation, industry, employer type and marital status are nominal (no order),
  so each category becomes its own 0/1 column. `handle_unknown="ignore"` protects against a category that appears only in the test set.
* **Numerical features → StandardScaler** (z = (x − mean) / std). KNN measures **distances** between people, so a feature with large numbers (age² is in thousands)
  would dominate one with small numbers (`work_from_home` is 0/1) — scaling puts them on a comparable scale (Week 5: scaling is mandatory for KNN). Tree models do not need scaling, but it does not harm them,
  so one shared preprocessor is used for all models.
* `SCHL` (education 1–24) and `ENG` (0–4) are **ordinal**, so they stay as numbers and keep their natural order.

The preprocessor is a `ColumnTransformer` — the same pattern as Seminar 4. It is **fitted on the training data only**;
in Section 4 it is placed inside a `Pipeline` together with each model, so during cross-validation it is re-fitted on each training fold.
`sparse_output=False` returns an ordinary table of numbers (a dense array), which is easier to inspect and much faster for the tree models.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
])

# Fit on the training set only, then check the result
X_train_prepared = preprocessor.fit_transform(X_train)
print(f"Features before preprocessing: {X_train.shape[1]}")
print(f"Features after preprocessing:  {X_train_prepared.shape[1]} "
      f"({len(NUM_FEATURES)} scaled numerical + {X_train_prepared.shape[1] - len(NUM_FEATURES)} one-hot columns)")

In [ ]:
# Check: after scaling, numerical features in the training set have mean ~0 and std ~1
scaler = preprocessor.named_transformers_["num"]          # the fitted StandardScaler inside the preprocessor
scaled = pd.DataFrame(scaler.transform(X_train[NUM_FEATURES]), columns=NUM_FEATURES)
scaled.describe().loc[["mean", "std"]].round(2)

### 3.7 Data preparation summary

| Step | Result |
|---|---|
| Inflation adjustment | wages in 2024 dollars |
| Error removal | rows with hourly wage < \$7.25 removed (≈3.9%) |
| Target | `log_wage` (skewness ≈ 0) |
| Missing values | filled with their structural meaning; 0 missing values left |
| Feature engineering | 25 occupation groups, 17 industry sectors, `age_sq`, `work_from_home` |
| Features | 8 numerical/ordinal + 4 categorical; protected attributes excluded |
| Split | 80/20, stratified by wage decile |
| Preprocessing | StandardScaler + OneHotEncoder in a ColumnTransformer, fitted on train only |

## 4. Model training
Four models from the module are trained and compared with a simple baseline:

| Model | Covered in | Why it is included |
|---|---|---|
| Dummy (mean) baseline | Seminar 6 | the "no model" reference — every real model must beat it |
| Linear Regression | Week 4, Seminar 5 | simple, interpretable, assumes a linear relationship |
| KNN Regressor | Week 5, Seminar 6 | non-linear, predicts from the most similar people |
| Random Forest | Week 6, Seminar 7 | bagging of deep trees — reduces variance, captures interactions |
| XGBoost | Week 6, Seminar 7 | boosting of shallow trees — reduces bias, often the most accurate |

Each model is wrapped in a `Pipeline(preprocessor → model)`, so the scaler and encoder are re-fitted inside every
cross-validation fold and the test set is never touched in this section.

### 4.1 Tuning approach
* **Hyperparameters** are chosen with `GridSearchCV` and **5-fold cross-validation** (Week 5, Seminar 6): the training data is split into
  5 parts, each combination is trained on 4 parts and scored on the 5th, and the average score decides the best combination.
* **Scoring:** mean absolute error (MAE) of the log-wage, as in Seminar 6 (`scoring="neg_mean_absolute_error"` — scikit-learn
  maximises scores, so MAE is used with a minus sign).
* **Tuning sample:** KNN has to compare every person with every other person, so a full grid search on 71k rows would take a long time.
  The search is therefore run on a **random sample of 20,000 training rows** (data reduction by sampling, Week 3). The best
  settings are then used to train each model on the **full training set**.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

FEATURES = NUM_FEATURES + CAT_FEATURES

# Random sample of the training set used for hyperparameter tuning
tune_df = train_df.sample(20_000, random_state=RANDOM_STATE)
X_tune, y_tune = tune_df[FEATURES], tune_df[TARGET]

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}      # cross-validation scores of every model
models = {}          # final models trained on the full training set

In [ ]:
def cross_validate_model(name, pipeline):
    # 5-fold cross-validation on the tuning sample: average MAE and R² over the 5 folds
    mae = -cross_val_score(pipeline, X_tune, y_tune, cv=kfold, scoring="neg_mean_absolute_error").mean()
    r2 = cross_val_score(pipeline, X_tune, y_tune, cv=kfold, scoring="r2").mean()
    cv_results[name] = {"CV MAE (log)": mae, "CV R²": r2}
    print(f"{name}: CV MAE = {mae:.4f}   CV R² = {r2:.3f}")


def tune_model(name, model, param_grid):
    # 1) grid search with 5-fold CV on the tuning sample
    pipeline = Pipeline([("prep", preprocessor), ("model", model)])
    search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring="neg_mean_absolute_error", n_jobs=-1)
    search.fit(X_tune, y_tune)
    print(f"Best parameters: {search.best_params_}")

    # 2) cross-validation scores of the best settings (MAE and R²)
    best = search.best_estimator_
    cross_validate_model(name, best)

    # 3) final training of the best settings on the full training set
    best.fit(X_train, y_train)
    models[name] = best
    return search

### 4.2 Baseline: predict the average wage
`DummyRegressor(strategy="mean")` ignores all features and predicts the average log-wage for everyone.
Its R² is 0 by definition — this is the reference point.

In [ ]:
baseline = Pipeline([("prep", preprocessor), ("model", DummyRegressor(strategy="mean"))])
cross_validate_model("Baseline (mean)", baseline)
models["Baseline (mean)"] = baseline.fit(X_train, y_train)

### 4.3 Linear Regression
Linear Regression has no hyperparameters to tune (Week 4): it finds the coefficients that minimise the squared error.
It is trained directly and scored with the same cross-validation.

In [ ]:
linear = Pipeline([("prep", preprocessor), ("model", LinearRegression())])
cross_validate_model("Linear Regression", linear)
models["Linear Regression"] = linear.fit(X_train, y_train)

### 4.4 KNN Regressor
KNN predicts a person's wage as the average wage of the *k* most similar people in the training set (Week 5). Tuned:
* `n_neighbors` (k) — small k follows noise (overfitting), large k over-smooths (underfitting);
* `weights` — every neighbour counts equally (`uniform`) or closer neighbours count more (`distance`);
* `metric` — Euclidean or Manhattan distance.

This is the same grid search as in Seminar 6 (Diamonds). Scaling (Section 3.6) is essential here because KNN works with distances.

In [ ]:
knn_grid = {
    "model__n_neighbors": [5, 10, 20, 30, 50],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"],
}
knn_search = tune_model("KNN", KNeighborsRegressor(), knn_grid)

In [ ]:
# How does the error change with k? (uniform weights, Euclidean distance)
results = pd.DataFrame(knn_search.cv_results_)
curve = results[(results["param_model__weights"] == "uniform") & (results["param_model__metric"] == "euclidean")]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(curve["param_model__n_neighbors"], -curve["mean_test_score"], marker="o", color=BLUE)
ax.set_xlabel("Number of neighbours k"); ax.set_ylabel("CV MAE (log-wage)")
ax.set_title("KNN: validation error for different k")
plt.tight_layout(); plt.show()

The error is high for small *k* (the prediction follows a few individual neighbours — overfitting), lowest around *k* = 20–30, and rises again
for large *k* (the prediction averages too many dissimilar people — underfitting). This is the bias–variance trade-off from Week 5.

### 4.5 Random Forest
A Random Forest trains many deep decision trees on bootstrap samples and averages them (bagging, Week 6). Each split considers
only a random subset of features (`max_features="sqrt"`, as in the lecture), which makes the trees different from each other. Tuned:
* `n_estimators` — number of trees;
* `max_depth` — maximum depth of each tree (`None` = grow until leaves are pure);
* `min_samples_leaf` — minimum people in a leaf (pre-pruning against overfitting).

In [ ]:
rf_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_leaf": [1, 5],
}
rf_search = tune_model("Random Forest",
                       RandomForestRegressor(max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1),
                       rf_grid)

### 4.6 XGBoost
XGBoost builds shallow trees **one after another**, each correcting the errors of the previous ones (boosting, Week 6). Tuned:
* `learning_rate` — how much each new tree contributes (smaller = more careful learning);
* `n_estimators` — number of boosting rounds;
* `max_depth` — depth of each tree (3–6 is typical for boosting, as in the lecture).

In [ ]:
xgb_grid = {
    "model__learning_rate": [0.05, 0.1],
    "model__n_estimators": [200, 400],
    "model__max_depth": [3, 5],
}
xgb_search = tune_model("XGBoost", XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1), xgb_grid)

### 4.7 Cross-validation comparison
All models were scored with the same 5 folds of the same tuning sample, so the comparison is fair.
The test set is still untouched — it is used once, in Section 5.

In [ ]:
cv_table = pd.DataFrame(cv_results).T.sort_values("CV MAE (log)")
display(cv_table.style.format("{:.4f}"))

fig, ax = plt.subplots(figsize=(7, 3.2))
order = cv_table.sort_values("CV R²")
ax.barh(order.index, order["CV R²"], color=BLUE, height=0.6)
for y, v in enumerate(order["CV R²"]):
    ax.text(v + 0.005, y, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlabel("Cross-validated R² (higher is better)"); ax.set_xlim(0, 0.7)
ax.set_title("Model comparison (5-fold CV on the training sample)")
plt.tight_layout(); plt.show()

**Findings:**
* **Every model clearly beats the baseline** (CV MAE ≈ 0.58 → 0.38–0.40, R² 0 → about 0.5), so the features carry real information about wages.
* **Linear Regression** (R² ≈ 0.51) is a strong simple model: most of the wage pattern is roughly linear in log-wage.
* **KNN** is only slightly better than Linear Regression, even after tuning.
* The **tree ensembles (Random Forest and XGBoost)** score best: they capture non-linear effects and interactions
  (e.g. the value of education differs between occupations) without these being engineered by hand.
* The differences between the top models are small. The final choice is confirmed on the **unseen test set** in Section 5.
* An R² of about 0.55 is typical for individual wage data: pay also depends on things that are not in the survey
  (employer, skills, experience in the role, negotiation), so no model can explain all of the variation.

## 5. Model evaluation on the test set
The test set (17,788 people) was separated in Section 3.5 and has not been used for any decision so far.
It is used here **once** to measure how well each model works on people it has never seen.

### 5.1 Evaluation metrics
The regression metrics from Week 4 / Seminar 5–6 are used:

* **MAE** (mean absolute error) — the average size of the error;
* **RMSE** (root mean squared error) — like MAE, but large errors are penalised more;
* **R²** — the share of the variation in (log) wages that the model explains (0 = no better than the average, 1 = perfect).

They are calculated on the log-wage (the scale the models were trained on). Because log-dollars are hard to interpret,
the predictions are also converted back to dollars with `np.exp()` and two extra measures are reported:
**MAE in dollars** and the **median absolute percentage error** — "for a typical person the prediction is X% away from the true wage".

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(model, X, y_log):
    # Metrics on the log scale and, after converting back with exp(), in dollars
    pred_log = model.predict(X)
    true_wage, pred_wage = np.exp(y_log), np.exp(pred_log)
    return {
        "MAE (log)":  mean_absolute_error(y_log, pred_log),
        "RMSE (log)": np.sqrt(mean_squared_error(y_log, pred_log)),
        "R²":         r2_score(y_log, pred_log),
        "MAE ($)":    mean_absolute_error(true_wage, pred_wage),
        "Median % error": np.median(np.abs(pred_wage - true_wage) / true_wage) * 100,
    }

test_results = pd.DataFrame({name: evaluate(model, X_test, y_test) for name, model in models.items()}).T
test_results = test_results.sort_values("MAE (log)")
display(test_results.style.format({"MAE (log)": "{:.4f}", "RMSE (log)": "{:.4f}", "R²": "{:.3f}",
                                   "MAE ($)": "${:,.0f}", "Median % error": "{:.1f}%"}))

### 5.2 Overfitting check: train vs test
A model that has memorised the training data scores much better on train than on test (overfitting, Week 5).
A model that is too simple scores poorly on both (underfitting).

In [ ]:
fit_check = pd.DataFrame({
    "Train R²": {name: r2_score(y_train, model.predict(X_train)) for name, model in models.items()},
    "Test R²":  test_results["R²"],
})
fit_check["Gap"] = fit_check["Train R²"] - fit_check["Test R²"]
display(fit_check.loc[test_results.index].style.format("{:.3f}"))

**How to read this table:**
* **Linear Regression** has almost no gap — a simple model cannot memorise the data.
* **KNN** with `weights="distance"` and **Random Forest** with fully grown trees score very high on train (each training person is
  their own nearest neighbour / sits in a pure leaf), so a large gap is expected. What matters is that their **test** score is still good,
  i.e. the cross-validation correctly chose settings that generalise.
* **XGBoost** builds shallow trees that each make a small correction, so its gap is usually much smaller than Random Forest's.

### 5.3 Final model
The final model is the one with the **best cross-validation score in Section 4** — the choice was made *before* looking at the test set.
The test results above confirm (or not) that choice on unseen data.

In [ ]:
final_name = cv_table.index[0]          # best model by cross-validated MAE (Section 4.7)
final_model = models[final_name]
print(f"Final model: {final_name}")
print(f"Test performance: MAE (log) = {test_results.loc[final_name, 'MAE (log)']:.4f}, "
      f"R² = {test_results.loc[final_name, 'R²']:.3f}, "
      f"MAE = ${test_results.loc[final_name, 'MAE ($)']:,.0f}, "
      f"median error = {test_results.loc[final_name, 'Median % error']:.1f}%")
print(f"Best model on the test set: {test_results.index[0]}")

### 5.4 Actual vs predicted wages
If the predictions were perfect, all points would lie on the diagonal line.

In [ ]:
test_pred_log = final_model.predict(X_test)
sample = np.random.RandomState(RANDOM_STATE).choice(len(y_test), 3000, replace=False)   # 3,000 points keep the plot readable

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(np.exp(y_test.iloc[sample]), np.exp(test_pred_log[sample]), s=6, alpha=0.3, color=BLUE)
axes[0].plot([1_000, 700_000], [1_000, 700_000], color=ORANGE, lw=2, label="perfect prediction")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("Actual wage ($, log scale)"); axes[0].set_ylabel("Predicted wage ($, log scale)")
axes[0].set_title(f"{final_name}: actual vs predicted (test set)"); axes[0].legend()

residuals = y_test - test_pred_log           # error on the log scale
axes[1].hist(residuals, bins=60, color=BLUE, edgecolor="white", linewidth=0.3)
axes[1].axvline(0, color=ORANGE, lw=2)
axes[1].set_xlabel("Residual = actual − predicted (log-wage)"); axes[1].set_ylabel("Number of people")
axes[1].set_title("Distribution of errors")
plt.tight_layout(); plt.show()

**Findings:** the points follow the diagonal, but the cloud is **flatter than the line**: the model over-predicts the lowest wages and
under-predicts the highest ones. This "regression to the mean" is typical when important information is missing — the model cannot know
who will be the exceptionally well-paid person in a given job, so it predicts closer to the group's typical wage.
The errors are centred on zero (no systematic bias) and roughly symmetric on the log scale.

### 5.5 Which features matter most? (permutation importance)
**Permutation importance** (Seminar 7) measures how much the test error grows when the values of one feature are randomly shuffled:
if shuffling a feature destroys the accuracy, the model relies on it. It works for any model and is calculated on the original
12 features (not on the 58 one-hot columns), so the result is easy to read.

In [ ]:
from sklearn.inspection import permutation_importance

perm_sample = np.random.RandomState(RANDOM_STATE).choice(len(X_test), 5000, replace=False)   # 5,000 test rows for speed
perm = permutation_importance(final_model, X_test.iloc[perm_sample], y_test.iloc[perm_sample],
                              scoring="neg_mean_absolute_error", n_repeats=5, random_state=RANDOM_STATE)

importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(importance.index, importance.values, color=BLUE, height=0.6)
ax.set_xlabel("Increase in MAE (log-wage) when the feature is shuffled")
ax.set_title(f"Permutation importance ({final_name}, test set)")
plt.tight_layout(); plt.show()
importance.sort_values(ascending=False).round(4).to_frame("importance")

**Findings:** education (`SCHL`), occupation, weeks worked and usual hours are the most important features (the exact order can differ
slightly between models), followed by industry. Age appears twice (`AGEP` and `age_sq`), which splits its importance between the two columns.
This matches the EDA (Section 2.7–2.8) and labour-economics theory: **how much you studied, what you do and how much you work** drive pay.
Commute time and working from home matter least.

### 5.6 Fairness check
Sex and race/ethnicity were **not** used as features (Section 3.4). The question here is whether the model's errors are still different
across groups. For each group the table shows:
* **Median error (%)** = the middle value of (predicted − actual) / actual. Positive = the model typically **over-predicts** this group's wages,
  negative = typically **under-predicts** (the median is used because a few very low actual wages make percentage averages explode);
* **Median % error** = typical *size* of the error, ignoring its direction.

In [ ]:
fair = test_df[["SEX", "RAC1P", "HISP"]].copy()
fair["sex"] = fair["SEX"].map(codes.SEX)
fair["race_eth"] = codes.race_ethnicity(fair["RAC1P"], fair["HISP"])
fair["actual"] = np.exp(y_test)
fair["predicted"] = np.exp(test_pred_log)
fair["error %"] = (fair["predicted"] - fair["actual"]) / fair["actual"] * 100

def group_errors(col):
    g = fair.groupby(col)
    return pd.DataFrame({
        "people": g.size(),
        "median actual ($)": g["actual"].median(),
        "median predicted ($)": g["predicted"].median(),
        "median error (%)": g["error %"].median(),
        "median % error": g["error %"].apply(lambda e: e.abs().median()),
    })

fmt = {"median actual ($)": "${:,.0f}", "median predicted ($)": "${:,.0f}", "median error (%)": "{:+.1f}%", "median % error": "{:.1f}%"}
display(group_errors("sex").style.format(fmt))
display(group_errors("race_eth").style.format(fmt))

**Findings:** because sex is not a feature, the model predicts wages for women and men **from their jobs, education and hours only**.
It therefore typically **over-predicts women's wages and under-predicts men's** (the median predicted wage of women is above their median
actual wage, and the opposite for men): part of the real pay gap is *not* explained by the features in the model. A similar pattern appears for
Black and Hispanic workers, whose wages are over-predicted more than those of White and Asian workers. This is an important result for the ethics discussion:
* used for **salary benchmarking**, the model recommends pay based on the job and qualifications, not on sex — which is the desired behaviour;
* the gap between actual and predicted wages for women is an estimate of the **unexplained pay gap** (the part not explained by occupation,
  industry, education and hours) — exactly what a pay-equity audit looks for;
* the typical size of the error (median % error) is similar across groups, so the model is not much less accurate for any one group.

### 5.7 Saving the final model
The whole pipeline (preprocessing + model) is saved with `joblib`, so the Streamlit app can load it and make predictions
without retraining. The result tables of this section are saved as CSV files for the app's evaluation page.

In [ ]:
import joblib

MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(final_model, MODEL_DIR / "wage_model.joblib")
print(f"Saved {final_name} pipeline to models/wage_model.joblib "
      f"({(MODEL_DIR / 'wage_model.joblib').stat().st_size / 1e6:.1f} MB)")

# Save the result tables so the Streamlit app can show them without retraining
cv_table.to_csv(MODEL_DIR / "cv_results.csv")
test_results.to_csv(MODEL_DIR / "test_results.csv")
fit_check.to_csv(MODEL_DIR / "fit_check.csv")
importance.sort_values(ascending=False).to_frame("importance").to_csv(MODEL_DIR / "importance.csv")
group_errors("sex").to_csv(MODEL_DIR / "fairness_sex.csv")
group_errors("race_eth").to_csv(MODEL_DIR / "fairness_race.csv")
print("Saved result tables to models/*.csv")

# Quick check: load it back and predict one person from the test set
loaded = joblib.load(MODEL_DIR / "wage_model.joblib")
example = X_test.iloc[[0]]
print(f"Example - predicted wage: ${np.exp(loaded.predict(example)[0]):,.0f}, actual wage: ${np.exp(y_test.iloc[0]):,.0f}")

### 5.8 Evaluation summary

| Question | Answer |
|---|---|
| Do the models learn anything? | Yes — every model beats the mean baseline by a wide margin |
| Which model is best? | The model chosen by cross-validation is confirmed on the test set (see 5.3) |
| How accurate is it? | It explains a little over half of the variation in log-wages; for a typical person the prediction is within the median % error shown in 5.1 |
| Overfitting? | Tree ensembles fit the training data closely, but their test scores remain the best, so they generalise |
| What drives wages? | Occupation, education, weeks worked, age and hours |
| Is it fair? | Errors are of similar size across groups; women's wages are over-predicted, revealing a pay gap not explained by job and qualifications |